# 01: Managed vs. External Tables

**Exam objective:** Differentiate between managed and external tables in Unity 
Catalog and perform basic operations (create, modify, delete, and convert between 
managed and external tables) on them.

**Free Edition note:** External tables require an external location and storage 
credential, neither of which can be created on Free Edition. The external-table 
portions of this notebook will be run to observe the expected errors and document 
the syntax; full hands-on practice requires a workspace with admin access to cloud 
storage.


In [0]:
USE CATALOG certprep;
USE SCHEMA governance;

## Managed Table Example

In [0]:
-- Create a simple managed table
CREATE TABLE managed_demo_01 (
    id INT,
    name STRING, 
    created_at TIMESTAMP
);

In [0]:
-- Insert a few rows
INSERT INTO managed_demo_01 VALUES 
    (0, "Eoin Carden", current_timestamp()),
    (1, 'Acheron', current_timestamp()),
    (2, 'Athanasius', current_timestamp()),
    (3, 'Doran', current_timestamp()),
    (4, 'Victor', current_timestamp());

In [0]:
-- Inspect Table Metadata
-- Take note of Type, Location, Provicder, Table Properties
DESCRIBE EXTENDED managed_demo_01;

### Describe Extended
- Type: MANAGED
- Location: null
- Provider: delta

In [0]:
-- Add a column to modify the table
ALTER TABLE managed_demo_01 ADD COLUMN character_intro_order INT;

In [0]:
-- Update the new column for existing rows
UPDATE managed_demo_01 SET character_intro_order = id + 1
WHERE id IS NOT NULL;

-- View data
SELECT * FROM managed_demo_01
ORDER BY id ASC;

### Dropping a Managed Table
Dropping a managed table actually drops the data. An external table's data is actually located elsewhere, such as with an ADLS Gen 2 container/folder. Dropping an external data only drops the metadata associated with that table, but no data is truly lost. Dropping a managed table is a true table drop.

In [0]:
-- Drop the table. 
DROP TABLE managed_demo_01;

In [0]:
-- Querying of course fails at this point. Will result in a TABLE_OR_VIEW_NOT_FOUND error
SELECT * FROM managed_demo_01;

## External Table Example

Syntax to create an external table

```
CREATE TABLE external_demo_01 (
    id INT,
    name STRING, 
    created_at TIMESTAMP
)
LOCATION 'abfss://CONTAINER@STORAGE_ACCOUNT.dfs.core.windows.net/external_demo_01/';
```

The above code will fail given that there is not a parent external location.

See Dropping Managed Tables above for note on dropping external tables.

Syntax for converting a managed table to external (Databricks Runtime 14.3+)

`ALTER TABLE <table_name> SET TBLPROPERTIES ('external' = 'true');`
 
Syntax for converting external to managed

`ALTER TABLE <table_name> SET TBLPROPERTIES ('external' = 'false');`

## Questions to Review

1. What are the three things Unity Catalog manages for a managed table that it does not manage for an external table?
2. If two different external tables in different schemas point to the same storage path, what happens if you drop one?
3. You inherit a project where data lives in an ADLS container that other non-Databricks tools also read from. Which table type do you choose and why?
4. What's the practical difference between a managed table and a Delta table? (Trick question — be precise.)

Answers

1. Storage location, data lifecycle, Optimization
    - Storage location: UC chooses where the data lives in the metastore's managed storage, instead of specifying the location. Ex: Metastore or Catalog managed storage container in ADLS.
    - Data lifecycle: DROP TABLE removes both metadata and data files for managed; only metadata for external.
    - Optimization: UC can run predictive optimization, automatic OPTIMIZE/VACUUM, and liquid clustering features on managed tables that it cannot on external tables (UC does not own the data files)
2. UC tables utilize a 3-part naming convention of catalog.schema.table, therefore only the specified external table is dropped. The source ADLS data files remain unaffected because they are external. The other external table likewise remains unaffected unless specified with a full qualified name. 
3. External table, because they live in storage you control, in a location you choose, and other tools can read those files directly without going through UC. An external path is more stable, and dropping an external table in Databricks does not delete the data in ADLS. Managed tables live in storage controlled by UC, in a location chosen by UC, with access mediated through UC's governance layer. A managed path is more likely to change, and dropping a managed table will delete the underlying data files.
4. Delta is a storage format, namely Parquet files + a transaction log. Managed vs External tables is a UC table-type distinction about who owns the data lifecycle. One can have a managed or external delta table, even managed or external non-delta tables (such as external over Parquet, CSV, or JSON files).